# Build the merged best-paths dataset

## 1. What this notebook does

Combines the **greedy** and **beam (PPO)** solutions into one file,
`merged_best_paths.jsonl` — one record per presentation (all 17,635), each with
its **shortest known path stored as the state trajectory** (`r1,r2` strings at
every step). This is the single, uniform format used to train the d-o-t model on
intermediate states.

Greedy paths are already states; **beam paths are packed moves**, so we replay
them through the env once to recover their intermediate states (that replay also
validates every beam solve). Only the ~7,290 beam-won paths are replayed.

**Runtime: CPU is fine** — no GPU needed (no policy network, no beam search, just
env stepping on tiny int8 arrays). Just `Runtime → Run all`.

## 2. Clone the repo (branch `test/eda`)

In [ ]:
import os
REPO = '/content/ACSolverX'
if not os.path.isdir(REPO):
    os.system(f'git clone -b test/eda https://github.com/Avi161/ACSolverX.git {REPO}')
%cd /content/ACSolverX
!git pull --ff-only

## 3. Install dependencies (CPU JAX 0.6.0)
Takes a couple of minutes. Installs the repo's base deps plus CPU JAX (no CUDA).

In [ ]:
%pip install -q -r requirements.txt
%pip install -q "jax==0.6.0"
print('dependencies installed (CPU JAX 0.6.0)')

## 4. Mount Google Drive + locate the harvest JSONL

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

# The beam-harvest output on your Drive (edit ONLY if your path differs):
JSONL = '/content/drive/MyDrive/beam_harvest_greedy_solved/pilot_results.jsonl'
assert os.path.exists(JSONL), f'not found: {JSONL}  <-- edit JSONL to point at your pilot_results.jsonl'

# Outputs go straight to Drive so they survive a Colab disconnect:
OUT_JSONL = '/content/drive/MyDrive/beam_harvest_greedy_solved/merged_best_paths.jsonl'
OUT_INDEX = '/content/drive/MyDrive/beam_harvest_greedy_solved/merged_best_paths_index.csv'
print('harvest JSONL:', JSONL)
print('will write   :', OUT_JSONL)

## 5. Regenerate env initial states (`data/greedy_all.txt`)
The env replays beam paths by line index, so it needs this file (no JAX; fast).

In [ ]:
!python scripts/csv_to_initial_states.py

## 6. Smoke-test the beam replay (50 paths)
Replays 50 beam-won paths and asserts each reaches the trivial presentation.
Writes nothing — it just proves the replay works before the full pass.

In [ ]:
!python scripts/build_merged_paths.py --jsonl "{JSONL}" --limit 50

## 7. Full run → `merged_best_paths.jsonl` (writes to Drive)
Expands all ~7,290 beam-won paths to states, merges with greedy, writes the
single-format dataset + a slim index CSV. Replay-validates every beam solve.

In [ ]:
!python scripts/build_merged_paths.py --jsonl "{JSONL}" --out_jsonl "{OUT_JSONL}" --out_index "{OUT_INDEX}"

## 8. Peek at the result (intermediate states + d-o-t)

In [ ]:
import json, collections
recs = {}
with open(OUT_JSONL) as f:
    for line in f:
        r = json.loads(line); recs[r['idx']] = r
print('presentations:', len(recs))
print('best_source  :', dict(collections.Counter(r['best_source'] for r in recs.values())))

bw = next(r for r in recs.values() if r['best_source'] == 'beam')
sp = bw['best_path']
print(f"\nexample beam-won idx {bw['idx']} (len {bw['best_path_length']}) - intermediate states + d-o-t:")
for t in range(0, len(sp), 2):
    print(f"  step {t//2:2d}: ({sp[t]}, {sp[t+1]})   d-o-t = {bw['best_path_length'] - t//2}")